# Import

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and Scale Dataset

## Load

In [24]:
iris = load_iris()
X = iris.data
y = iris.target

print("Feature shape:", X.shape)
print("Label shape:", y.shape)


Feature shape: (150, 4)
Label shape: (150,)


## Split

In [12]:
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
    X_test_val,
    y_test_val,
    test_size=0.5,   # 0.2 of 0.8 = 0.16 of the full dataset
    random_state=42,
    stratify=y_test_val
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 112
Validation size: 19
Test size: 19


## Standardize

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


# Dataloader
A DataLoader is used to load the dataset in small batches during training. It can also shuffle the training data, which helps the model learn better. In PyTorch, it is the standard way to iterate over data efficiently and cleanly.

In [14]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# Create datasets and dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model

In [15]:
class base_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        # A simple network with one hidden layer
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [16]:
class advanced_MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, dropout_rate=0.3):
        super().__init__()

        # A simple network with one hidden layer and dropout
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [17]:
base_model = base_MLP(4, 16, 3)
advanced_model = advanced_MLP(4, 16, 3, 0.3)

In [18]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
base_model_optimizer = optim.Adam(base_model.parameters(), lr=0.01)
advanced_model_optimzer = optim.Adam(advanced_model.parameters(),lr=0.01)

# Training

## Base model

In [19]:
num_epochs = 100

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    base_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in progress_bar:
        # Forward pass
        train_outputs = base_model(batch_X)
        train_loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        base_model_optimizer.zero_grad()
        train_loss.backward()
        base_model_optimizer.step()

        # Collect training statistics
        train_loss_sum += train_loss.item() * batch_X.size(0)
        train_predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (train_predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        progress_bar.set_postfix({
            "batch_loss": f"{train_loss.item():.4f}"
        })

    train_loss_epoch = train_loss_sum / train_total
    train_acc_epoch = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    base_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = base_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += val_loss.item() * batch_X.size(0)
            val_predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (val_predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss_epoch = val_loss_sum / val_total
    val_acc_epoch = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss_epoch:.4f} | "
        f"Train Acc: {train_acc_epoch:.4f} | "
        f"Val Loss: {val_loss_epoch:.4f} | "
        f"Val Acc: {val_acc_epoch:.4f}"
    )




Epoch 1/100:   0%|          | 0/7 [00:00<?, ?it/s]c:\Users\chris\Documents\GitHub\ML_DL\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: cudaGetDeviceCount() returned cudaErrorNotSupported, likely using older driver or on CPU machine (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:88.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch [1/100] | Train Loss: 1.1502 | Train Acc: 0.2679 | Val Loss: 1.0234 | Val Acc: 0.3684


Epoch [2/100] | Train Loss: 0.8694 | Train Acc: 0.5714 | Val Loss: 0.8109 | Val Acc: 0.5789


Epoch [3/100] | Train Loss: 0.6503 | Train Acc: 0.7054 | Val Loss: 0.6306 | Val Acc: 0.8947


Epoch [4/100] | Train Loss: 0.5035 | Train Acc: 0.8125 | Val Loss: 0.5033 | Val Acc: 0.8947


...

Epoch [96/100] | Train Loss: 0.0345 | Train Acc: 0.9821 | Val Loss: 0.0456 | Val Acc: 1.0000


Epoch [97/100] | Train Loss: 0.0349 | Train Acc: 0.9821 | Val Loss: 0.0366 | Val Acc: 1.0000


Epoch [98/100] | Train Loss: 0.0368 | Train Acc: 0.9821 | Val Loss: 0.0403 | Val Acc: 1.0000


Epoch [99/100] | Train Loss: 0.0352 | Train Acc: 0.9821 | Val Loss: 0.0487 | Val Acc: 1.0000


Epoch [100/100] | Train Loss: 0.0345 | Train Acc: 0.9821 | Val Loss: 0.0397 | Val Acc: 1.0000


In [20]:
from sklearn.metrics import classification_report, confusion_matrix

# -----------------------------
# 8. Final evaluation on the test set
# -----------------------------
base_model.eval()

with torch.no_grad():
    test_outputs = base_model(X_test)
    test_predictions = torch.argmax(test_outputs, dim=1)
    test_accuracy = (test_predictions == y_test).float().mean()

print("\nFinal Test Accuracy:", test_accuracy.item())

# Convert tensors to numpy arrays for scikit-learn
y_true = y_test.numpy()
y_pred = test_predictions.numpy()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=iris.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Final Test Accuracy: 0.8947368264198303

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         6
  versicolor       0.83      0.83      0.83         6
   virginica       0.86      0.86      0.86         7

    accuracy                           0.89        19
   macro avg       0.90      0.90      0.90        19
weighted avg       0.89      0.89      0.89        19

Confusion Matrix:
[[6 0 0]
 [0 5 1]
 [0 1 6]]


## Model with Dropout and Early stopping

In [21]:

num_epochs = 1000
patience = 100  # Stop if validation accuracy does not improve for 10 epochs


In [22]:

best_val_accuracy = 0.0
best_model_state = {
    key: value.detach().clone()
    for key, value in advanced_model.state_dict().items()
}
epochs_without_improvement = 0

for epoch in range(num_epochs):
    # -----------------------------
    # Training
    # -----------------------------
    advanced_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch_X, batch_y in train_progress:
        # Forward pass
        train_outputs = advanced_model(batch_X)
        loss = criterion(train_outputs, batch_y)

        # Backward pass and parameter update
        advanced_model_optimzer.zero_grad()
        loss.backward()
        advanced_model_optimzer.step()

        # Collect training statistics
        train_loss_sum += loss.item() * batch_X.size(0)
        predictions = torch.argmax(train_outputs, dim=1)
        train_correct += (predictions == batch_y).sum().item()
        train_total += batch_y.size(0)

        train_progress.set_postfix({
            "batch_loss": f"{loss.item():.4f}"
        })

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # -----------------------------
    # Validation
    # -----------------------------
    advanced_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            val_outputs = advanced_model(batch_X)
            val_loss = criterion(val_outputs, batch_y)

            val_loss_sum += loss.item() * batch_X.size(0)
            predictions = torch.argmax(val_outputs, dim=1)
            val_correct += (predictions == batch_y).sum().item()
            val_total += batch_y.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # -----------------------------
    # Early stopping
    # -----------------------------
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().clone()
            for key, value in advanced_model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break



Epoch [1/1000] | Train Loss: 1.0174 | Train Acc: 0.3929 | Val Loss: 0.9575 | Val Acc: 0.5789


Epoch [2/1000] | Train Loss: 0.8252 | Train Acc: 0.7143 | Val Loss: 0.7653 | Val Acc: 0.7368


Epoch [3/1000] | Train Loss: 0.6554 | Train Acc: 0.7500 | Val Loss: 0.6173 | Val Acc: 0.8421


Epoch [4/1000] | Train Loss: 0.4993 | Train Acc: 0.8393 | Val Loss: 0.3349 | Val Acc: 0.8421


Epoch [5/1000] | Train Loss: 0.4146 | Train Acc: 0.8571 | Val Loss: 0.4117 | Val Acc: 0.8421


...

Epoch [117/1000] | Train Loss: 0.0667 | Train Acc: 0.9732 | Val Loss: 0.0912 | Val Acc: 1.0000


Epoch [118/1000] | Train Loss: 0.0613 | Train Acc: 0.9911 | Val Loss: 0.1189 | Val Acc: 1.0000


Epoch [119/1000] | Train Loss: 0.0778 | Train Acc: 0.9643 | Val Loss: 0.0867 | Val Acc: 1.0000


Epoch [120/1000] | Train Loss: 0.0672 | Train Acc: 0.9732 | Val Loss: 0.0338 | Val Acc: 1.0000


Epoch [121/1000] | Train Loss: 0.0731 | Train Acc: 0.9732 | Val Loss: 0.0336 | Val Acc: 1.0000

Early stopping triggered.


In [23]:

# Load the best model
advanced_model.load_state_dict(best_model_state)

advanced_model.eval()
test_correct = 0
test_total = 0

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_outputs = advanced_model(batch_X)
        test_predictions = torch.argmax(test_outputs, dim=1)

        # Update accuracy statistics
        test_correct += (test_predictions == batch_y).sum().item()
        test_total += batch_y.size(0)

        # Store predictions and true labels
        all_y_true.extend(batch_y.numpy())
        all_y_pred.extend(test_predictions.numpy())

test_accuracy = test_correct / test_total
print("\nFinal Test Accuracy:", test_accuracy)

# Print classification report
print("\nClassification Report:")
print(classification_report(all_y_true, all_y_pred, target_names=iris.target_names))

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(all_y_true, all_y_pred))


Final Test Accuracy: 0.8947368421052632

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         6
  versicolor       1.00      0.67      0.80         6
   virginica       0.78      1.00      0.88         7

    accuracy                           0.89        19
   macro avg       0.93      0.89      0.89        19
weighted avg       0.92      0.89      0.89        19

Confusion Matrix:
[[6 0 0]
 [0 4 2]
 [0 0 7]]
